# Setup

In [ ]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# spark configuration
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" or os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = "" or os.getenv("SPARK_HOME")

# subsampling: percentage of commenting users to keep
USERS_PERCENTAGE = 0.1
assert 0 < USERS_PERCENTAGE <= 1, "invalid USER_PERCENTAGE, should be in (0, 1]"

# configuration
MIN_KW_FREQ = 5 # minimum frequency of keywords to be included in the profile (for the dictionary approach)
FEATURES_SIZE = 5000 # number of features, buckets of the hash (for the hashing approach)
LSH_NUM_HASHES = 100 # number of random hyperplanes (signatures/sketches)
LSH_BANDS = 10 # TODO: play with b and r (calculate the threshold for different values)
LSH_ROWS = 10
LSH_MAX_BUCKET_SIZE = 500 # maximum number of users/articles in the same bucket
RECOMMENDED_ARTICLES = 10 # number of articles to recommend to each user
assert LSH_NUM_HASHES == LSH_BANDS * LSH_ROWS, "invalid LSH_NUM_HASHES, must be equal to LSH_BANDS * LSH_ROWS"


In [ ]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)


In [ ]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j
import pyspark
ss = (pyspark.sql.SparkSession
    .builder
    .getOrCreate())
sc = ss.sparkContext


In [5]:
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"', # quotes " in the dataset are escaped ""
    "multiLine": "true",
    "mode": "PERMISSIVE",
}

full_articles = ss.read.options(**csv_options).csv("dataset/nyt-articles-2020.csv")
full_comments = ss.read.options(**csv_options).csv("dataset/nyt-comments-part0.csv") # TODO: this is a subset of all comments

print(f"Number of articles: {full_articles.count()}")
print(f"Number of comments: {full_comments.count()}")
full_articles.take(1), full_comments.take(1)


Number of articles: 16787
Number of comments: 500000


([Row(newsdesk='Editorial', section='Opinion', subsection=None, material='Editorial', headline='Protect Veterans From Fraud', abstract='Congress could do much more to protect Americans who have served their country from predatory for-profit colleges.', keywords="['Veterans', 'For-Profit Schools', 'Financial Aid (Education)', 'Frauds and Swindling', 'Colleges and Universities', 'Veterans Affairs Department', 'Federal Trade Commission', 'University of Phoenix', 'Career Education Corporation']", word_count=680, pub_date=datetime.datetime(2020, 1, 1, 0, 18, 54), n_comments=186, uniqueID='nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')],
 [Row(commentID=104387472, status='approved', commentSequence=104387472, userID=60215558, userDisplayName='magicisnotreal', userLocation='earth', userTitle=None, commentBody='Here is something I think is fraudulent that vets are subject to\n If you use your VA home loan option you have to pay higher interest rates regardless of your credit rating becua

In [ ]:
%pip install mmh3 # non-cryptographic hash, better distribution
# utility hash function: bytes | str -> signed int 32
def h(data):
    import mmh3
    return mmh3.hash(data)


In [6]:
# parse keywords column utility
def parse_keywords(row):
    import re
    if not row["keywords"]: return []
    kws = re.sub(r'\[|\]|\'|\"', '', row["keywords"].lower()).strip()
    if not kws: return []
    return [w.strip() for w in kws.split(",") if len(w.strip()) > 0]

# keep only relevant information for recommendations porpouse, convert from dataframe to RDD tuples
articles = (full_articles
            .select("uniqueID", "newsdesk", "section", "subsection", "keywords")
            .rdd
            .map(lambda row: (row["uniqueID"], row["newsdesk"], row["section"], row["subsection"], parse_keywords(row))))
comments = (full_comments
            .select("articleID", "userID")
            .rdd
            .map(lambda row: (row["articleID"], row["userID"])))

print(f"Number of articles: {articles.count()}")
print(f"Number of comments: {comments.count()}")
print(f"Number of users: {comments.map(lambda x: x[1]).distinct().count()}")
articles.take(1), comments.take(1)


Number of articles: 16787
Number of comments: 500000
Number of users: 91437


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   'Editorial',
   'Opinion',
   None,
   ['veterans',
    'for-profit schools',
    'financial aid (education)',
    'frauds and swindling',
    'colleges and universities',
    'veterans affairs department',
    'federal trade commission',
    'university of phoenix',
    'career education corporation'])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)])

## Utility Matrix

In [7]:
utility_matrix = comments.distinct() # ignore multiple comments from same user on same article
print(f"Number of entries in sparse utility matrix: {utility_matrix.count()}")
utility_matrix.take(5)


Number of entries in sparse utility matrix: 356283


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65691034),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65110053),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 66232009),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 73299044)]

## Articles Profile - via dictionary

In [8]:
# count frequencies
keywords = (articles
            .flatMap(lambda row: [(k, 1) for k in row[4]])
            .reduceByKey(lambda a, b: a + b))

# prune too rare keywords
pruned_keywords = (keywords
                   .filter(lambda x: x[1] >= MIN_KW_FREQ and len(x[0]) <= 100)
                   .map(lambda x: f"KEY {x[0]}")
                   .collect())

# extract all features
features = sorted(pruned_keywords + (articles
            .flatMap(lambda row: [f"{f} {row[i+1].lower().strip()}" for i, f in enumerate(["NEW", "SEC", "SUB"]) if row[i+1]])
            .filter(lambda x: len(x) < 100)
            .distinct()
            .collect()))

# features mapping dictionary, needs to be broadcasted to every node
features_mapping = {f: i for i, f in enumerate(features)}
print(f"Extacted {len(features_mapping)} features (broadcasting at most {((100 * 4) + 4) * len(features_mapping) // 1024} KB)")
broadcast_dict = sc.broadcast(features_mapping)

# build sparse vectors for profiles (article_id, feature_index)
def build_article_profile_dict(row):
    mapping = broadcast_dict.value
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        if f not in mapping: continue
        res.append((row[0], mapping[f]))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        if k not in mapping: continue
        res.append((row[0], mapping[k]))
    return res

articles_profile_dict = articles.flatMap(build_article_profile_dict)
print(f"Number of entries in sparse articles profile via dictionary: {articles_profile_dict.count()}")
articles_profile_dict.take(10)


Extacted 3555 features (broadcasting at most 1402 KB)
Number of entries in sparse articles profile via dictionary: 158079


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3395),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3465),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3193),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1131),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1097),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1158),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 628),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3194),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1086),
 ('nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c', 3399)]

## Articles Profile - via hashing

In [9]:
# build sparse vectors for profiles (article_id, feature_hash)
def build_article_profile_hash(row):
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        res.append((row[0], h(f) % FEATURES_SIZE))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        res.append((row[0], h(k) % FEATURES_SIZE))
    return res

articles_profile_hash = articles.flatMap(build_article_profile_hash)
print(f"Number of entries in sparse articles profile via hash: {articles_profile_hash.count()}")
articles_profile_hash.take(10)


Number of entries in sparse articles profile via hash: 185396


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 653),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2704),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2692),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1483),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 4497),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 260),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1675),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2683),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3196),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 598)]

### WARNING

From now on, the operations will be **independent** of how the profiles were computed (via dictionary or hashing).
Change the content of next cell to change approach: (`articles_profile_dict` or `articles_profile_hash`)

In [10]:
articles_profile = articles_profile_hash


## Users Profile

In [11]:
# build user profiles by joining utility matrix with article profiles, then counting feature frequencies for each user
users_profile = (utility_matrix
                     .join(articles_profile)
                     .map(lambda x: ((x[1][0], x[1][1]), 1))
                     .reduceByKey(lambda a, b: a + b))
print(f"Number of entries in sparse users profile: {users_profile.count()}")
users_profile.take(10)


Number of entries in sparse users profile: 2801463


[((5646097, 535), 36),
 ((5646097, 2881), 36),
 ((5646097, 4221), 36),
 ((86273619, 535), 11),
 ((86273619, 2881), 11),
 ((86273619, 4221), 11),
 ((61536727, 535), 9),
 ((61536727, 2881), 9),
 ((61536727, 4221), 9),
 ((41508509, 535), 20)]

## Computing Similarity - LSH using hyperplanes (sketches)

In [12]:
import random

# random vectors of +1 and -1 of dimensione FEATURES_SIZE
random_hyperplanes = [[random.choice([-1, 1]) for _ in range(FEATURES_SIZE)] for _ in range(LSH_NUM_HASHES)]
b_hyperplanes = sc.broadcast(random_hyperplanes)

def compute_sketch(row):
    item_id, sparse_features = row
    sketch = []
    for plane in b_hyperplanes.value:
        # dot product: if above or below the hyperplane
        # only considering the features that exist in the sparse vector
        dot_product = sum(weight * plane[feat] for feat, weight in sparse_features)
        # 1 if positive, 0 if negative
        sketch.append(1 if dot_product > 0 else 0)
    return (item_id, sketch)

# compute sketches ("equivalent" of signature) for users and articles
users_sketches = (users_profile
                  .map(lambda x: (x[0][0], ((x[0][1], x[1]))))
                  .groupByKey()
                  .mapValues(list)
                  .map(compute_sketch))
articles_sketches = (articles_profile
                  .map(lambda x: (x[0], ((x[1], 1))))
                  .groupByKey()
                  .mapValues(list)
                  .map(compute_sketch))

print(f"Number of users sketches: {users_sketches.count()}")
print(f"Number of articles sketches: {articles_sketches.count()}")
users_sketches.take(1), articles_sketches.take(1)


Number of users sketches: 91437
Number of articles sketches: 16787


([(13120442,
   [1,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    1])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   [0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
   

In [13]:
def lsh_buckets(row):
    item_id, sketch = row
    res = []
    for band in range(LSH_BANDS):
        band_signature = tuple(sketch[band*LSH_ROWS : (band+1)*LSH_ROWS])
        bucket_hash = h(f"b{band}_{band_signature}") # include band index to avoid collisions between different bands
        res.append((bucket_hash, item_id))
    return res

users_buckets = users_sketches.flatMap(lsh_buckets)
articles_buckets = articles_sketches.flatMap(lsh_buckets)

print(f"Total number of users buckets: {users_buckets.count()}")
print(f"Total number of articles buckets: {articles_buckets.count()}")

# buckets too big are not useful, too few information
valid_users_buckets = (users_buckets
                       .map(lambda x: (x[0], 1))
                       .reduceByKey(lambda a, b: a + b)
                       .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

valid_articles_buckets = (articles_buckets
                          .map(lambda x: (x[0], 1))
                          .reduceByKey(lambda a, b: a + b)
                          .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

# the join works as an intersection between keys
users_buckets = (users_buckets
                 .join(valid_users_buckets)
                 .map(lambda x: (x[0], x[1][0])))
articles_buckets = (articles_buckets
                    .join(valid_articles_buckets)
                    .map(lambda x: (x[0], x[1][0])))

print(f"Valid number of users buckets: {users_buckets.count()}")
print(f"Valid number of articles buckets: {articles_buckets.count()}")

users_buckets.take(10), articles_buckets.take(10)


Total number of users buckets: 914370
Total number of articles buckets: 167870
Valid number of users buckets: 592523
Valid number of articles buckets: 160863


([(1454101488, 13120442),
  (1454101488, 12503724),
  (1454101488, 48114248),
  (1454101488, 65915584),
  (1454101488, 65619826),
  (1454101488, 60314966),
  (1454101488, 85673042),
  (1454101488, 93235460),
  (1454101488, 46052572),
  (1454101488, 57879042)],
 [(1893601130, 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd'),
  (1893601130, 'nyt://article/38e2b98b-364e-5abc-950c-9ff699e52f2b'),
  (1893601130, 'nyt://article/c7c87c5e-a6c8-53af-a462-5954f1258be7'),
  (1893601130, 'nyt://article/63dcf8a6-7ff8-54ee-8b05-ee4fdafec233'),
  (1893601130, 'nyt://article/8128a13e-af9f-524f-9f47-ca173338fe2c'),
  (1893601130, 'nyt://article/8dfd7f72-4cc9-55cd-923b-4326b35e8d8c'),
  (1893601130, 'nyt://article/0cee0288-3096-5e9f-81e1-d2ae6995edb3'),
  (1893601130, 'nyt://interactive/c61c61c1-56d8-5256-b684-84a8d145430d'),
  (1893601130, 'nyt://article/a66d0418-f808-58be-86e7-220bb7129404'),
  (1893601130, 'nyt://article/680579dc-c682-54d1-a3be-cd8cc7f8ef8f')])

In [14]:
# no distinct no subtract no bucket pruning 20 bands x 5 rows = ~1b results in 3min 20s
# no distinct no subtract no bucket pruning 10 bands x 10 rows = ~50m results in 20s
# distinct no bucket pruning 10 bands x 10 rows = ~30m results in 6min 50s
# distinct + subtract (after bucket pruning) 10 bands x 10 rows = ~12m results in 4m 30s

lsh_recommendations = (users_buckets
                   .join(articles_buckets)
                   .map(lambda x: (x[1][0], x[1][1]))
                   .distinct()
                   .subtract(utility_matrix))

print(f"Number of recommendations that passed LSH filter: {lsh_recommendations.count()}")
lsh_recommendations.take(10)


Number of recommendations that passed LSH filter: 12625080


[(49821116, 'nyt://article/77c3dcef-12ba-56c4-9b63-161716f9e805'),
 (60444479, 'nyt://article/e231990b-8cd0-5f6f-a6af-45ae6894efd2'),
 (69024970, 'nyt://article/00485467-5483-59f4-b534-bb76ff3a2d4c'),
 (60572961, 'nyt://article/d9f7e1b5-44d3-598f-a7c1-16e9706336b0'),
 (106954221, 'nyt://article/32b2fac4-0ae9-5bef-938c-eb4b0a425b61'),
 (88596539, 'nyt://article/87c5321c-3bac-5cca-bab6-d2b1341ecd1b'),
 (86918913, 'nyt://article/a9abd0c4-593b-534a-a9cf-4d86569ff9b6'),
 (77738809, 'nyt://article/fd3a7dcd-7c4d-55d9-a582-0f2cac623f92'),
 (57387669, 'nyt://article/64637f65-f334-5093-952f-381f8d585a1e'),
 (89389393, 'nyt://article/40ead710-5f13-537a-bfd2-5836bb8ed5ce')]

In [15]:
# ~6m results in 4m 50s

# cosine similarity between sparse profiles, user_profile and article_profile should be dicts {feature_index: weight}
def cosine_similarity(user_profile, article_profile):
    dot_product = sum(user_profile.get(feat, 0) * weight for feat, weight in article_profile.items())
    user_norm = sum(weight ** 2 for weight in user_profile.values()) ** 0.5
    article_norm = sum(weight ** 2 for weight in article_profile.values()) ** 0.5
    if user_norm == 0 or article_norm == 0:
        return 0.0
    return dot_product / (user_norm * article_norm)

# calculate actual similarity by joining the profiles and applying
recommendations = (lsh_recommendations # (user_id, article_id)
                   .join(users_profile.map(lambda x: (x[0][0], (x[0][1], x[1]))).groupByKey().mapValues(dict)) # (user_id, (article_id, {profile}))
                   .map(lambda x: (x[1][0], (x[0], x[1][1]))) # (article_id, (user_id, {user_profile}))
                   .join(articles_profile.map(lambda x: (x[0], (x[1], 1))).groupByKey().mapValues(dict)) # (article_id, ((user_id, {user_profile}), {article_profile}))
                   .map(lambda x: (x[1][0][0], (x[0], cosine_similarity(x[1][0][1], x[1][1])))) # (user_id, (article_id, actual_sim))
                   .filter(lambda x: x[1][1] > 0.0))

print(f"Number of recommendations with actual cosine similarity: {recommendations.count()}")
recommendations.take(10)


Number of recommendations with actual cosine similarity: 6456243


[(39583521,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.681005224606999)),
 (64911096,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.6666666666666667)),
 (102939822,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.5892556509887896)),
 (52483077,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.6243905410544627)),
 (53665920,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.6666666666666667)),
 (85985883,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.21081851067789195)),
 (15263001,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.5892556509887896)),
 (40968198,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.0936585811581694)),
 (59865381,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.11322770341445959)),
 (7702371,
  ('nyt://article/60022a78-12bc-5ece-92a2-20f12cf1464e', 0.6666666666666667))]

In [16]:
# around 1m 20s

import heapq

user_recommendations = (recommendations
                        .map(lambda x: (x[0], (x[1][0], round(x[1][1], 5))))
                        .groupByKey()
                        .mapValues(lambda iter: heapq.nlargest(RECOMMENDED_ARTICLES, iter, key=lambda x: x[1])))

print(f"Number of users with at least one recommendation: {user_recommendations.count()}")
user_recommendations.take(10)


Number of users with at least one recommendation: 84129


[(79194330,
  [('nyt://article/68111fb7-b006-580a-b3ae-35d7601a748d', 0.59805),
   ('nyt://article/f815919f-c57f-5ced-bcda-7f4624134036', 0.53875),
   ('nyt://article/971afce9-6bba-592d-9807-f956888d566d', 0.4967),
   ('nyt://article/fb3b5c64-3799-5768-8142-11055f8c9700', 0.48631),
   ('nyt://article/f90d25e2-9b88-5c17-b980-64d84599cb08', 0.4833),
   ('nyt://article/60271bfc-eadb-513b-b1e2-32ad0291382c', 0.4833),
   ('nyt://article/aa7dd152-2127-50f1-88cf-e49b82e3ae7f', 0.4833),
   ('nyt://article/5c8fdeb8-9ec4-5ad7-b5a3-d6d22504352e', 0.4833),
   ('nyt://article/281e4be0-5576-5230-90e4-3c2c9514e1bc', 0.47951),
   ('nyt://article/22be24e7-b140-5997-a025-2f34e4ddc6fa', 0.47659)]),
 (98652420,
  [('nyt://article/843a4c72-6ba1-5f81-aa8f-5a859e7bfac5', 1.0),
   ('nyt://article/ba3268aa-d04a-525c-b478-07d59e3f79cb', 0.41812),
   ('nyt://article/3060cc94-f1a7-58cd-8c87-9ac02904630c', 0.40452),
   ('nyt://article/3bfb7c13-3289-5e91-9026-fb67022c6c04', 0.40202),
   ('nyt://article/19bba3cb-1f0